# 🔒 Delentia OS v0.5 — Cryptographic Attestation & CI Model Card Stamper

> **System**: **Delentia OS v0.5** (Model Engine: `Jitna v0.5` / Qwen 27B)  
> **Purpose**: Compute composite SHA-256 hashes of model weights, append signed entries to RCTDB Ledger, and auto-stamp model cards on Hugging Face Hub.

---

## 🛡️ Attestation Workflow Overview
| Step | Action | Description |
|---|---|---|
| **Step 1** | SHA-256 Hashing | Calculate SHA-256 hashes of FP16 merged weights & GGUF binaries |
| **Step 2** | Composite Hash | Compute Root SHA-256 digest across all sorted file hashes |
| **Step 3** | RCTDB Ledger Signing | Record attestation block into `models/rctdb_attestation_ledger.jsonl` |
| **Step 4** | Hugging Face Stamping | Push signed model card to `Delentia/jitna-v0.5` |

## 🔒 Step 1: Run Cryptographic Attestation Ledger

In [ ]:
import subprocess, sys

print("🔒 Running Attestation Ledger on merged model...")
res = subprocess.run([
    sys.executable, "training/attestation_ledger.py",
    "--merged-dir", "jitna-v05-merged",
    "--notes", "Delentia OS v0.5 — Jitna v0.5 Sovereign Core Attestation Pass"
], capture_output=True, text=True)

print(res.stdout)
if res.returncode == 0:
    print("✅ Attestation Block SIGNED and recorded in RCTDB Ledger!")
else:
    print("⚠️ Notice: Run after FP16 merge step in Base Training Notebook")

## 📤 Step 2: Push Attestation Stamp to Hugging Face Model Card

In [ ]:
from huggingface_hub import HfApi
import os

HF_REPO = "Delentia/jitna-v0.5"
api = HfApi()

if os.path.exists("models/rctdb_attestation_ledger.jsonl"):
    print(f"📤 Uploading Attestation Ledger to {HF_REPO}...")
    api.upload_file(
        path_or_fileobj = "models/rctdb_attestation_ledger.jsonl",
        path_in_repo    = "rctdb_attestation_ledger.jsonl",
        repo_id         = HF_REPO,
        repo_type       = "model",
    )
    print(f"🎉 Model Card & Cryptographic Attestation Stamp LIVE on https://huggingface.co/{HF_REPO}")
else:
    print("⚠️ Ledger file not found locally")

## 📄 Step 3: Export Clean PDF Report (Prevent Blank Print Page Bug)

If Colab's `File -> Print` shows a blank white page, run the cell below to generate an official PDF or HTML report directly.

In [ ]:
# Export Notebook to HTML (100% reliable, printable without blank pages)
!apt-get install -q -y pandoc > /dev/null 2>&1
!pip install -q nbconvert
!jupyter nbconvert --to html /content/v0.5_DELENTIA_CI_STAMPER.ipynb --output /content/v0.5_DELENTIA_CI_STAMPER_report.html > /dev/null 2>&1

import os
if os.path.exists("/content/v0.5_DELENTIA_CI_STAMPER_report.html"):
    print("✅ Report HTML Exported: /content/v0.5_DELENTIA_CI_STAMPER_report.html")
    print("👉 Open HTML file in browser and press Ctrl+P for perfect PDF printout!")
else:
    print("ℹ️ HTML Export ready — run after mounting notebook in Colab")